# 02 — Modélisation

Ce notebook compare trois modèles de régression pour prédire le prix des véhicules :

- DummyRegressor
- LinearRegression
- RandomForestRegressor

Les modèles seront entraînés sur les mêmes données et évalués avec les mêmes métriques.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/interim/vehicles_clean.csv")

df.head()

,YEAR,KM,BODYDOORS,PASSENGERS,MAKE,MODEL,BODYDESCF,DRIVETRAIN,FUEL,TRANSDESCF,COLORF,PRICE
0,2024,14243,4,5,Nissan,Versa,Berline,Traction avant,Essence,Automatique,Gris,18995
1,2021,58800,4,5,Ford,Escape,VUS,4 roues motrices,Essence,Automatique,Noir,18995
2,2019,85525,2,0,Chevrolet,Express Commercial Cutaway,Berline,Propulsion,Essence,Automatique,Blanc,26995
3,2023,60820,4,5,Hyundai,Venue,VUS,Traction avant,Essence,Automatique,Bleu,17995
4,2020,103329,4,5,Volkswagen,Jetta,Berline,Traction avant,Essence,Automatique,Argent,17495


In [3]:
df.shape

(543, 12)

## Séparation des variables explicatives et de la cible

In [4]:
X = df.drop(columns="PRICE")
y = df["PRICE"]

In [5]:
X.shape, y.shape

((543, 11), (543,))

## Séparation des données d'entraînement et de test

Les données sont séparées en deux groupes :

- 80 % pour entraîner les modèles ;
- 20 % pour évaluer leurs performances sur des données non utilisées pendant l'entraînement.

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [8]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (434, 11)
X_test  : (109, 11)
y_train : (434,)
y_test  : (109,)


## Préparation des variables

Les variables numériques sont conservées telles quelles.

Les variables catégorielles sont transformées avec `OneHotEncoder` afin de convertir les catégories en valeurs numériques utilisables par les modèles.

In [9]:
variables_numeriques = [
    "YEAR",
    "KM",
    "BODYDOORS",
    "PASSENGERS"
]

variables_categorielles = [
    "MAKE",
    "MODEL",
    "BODYDESCF",
    "DRIVETRAIN",
    "FUEL",
    "TRANSDESCF",
    "COLORF"
]

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "numeriques",
            "passthrough",
            variables_numeriques
        )
    ]
)

In [12]:
X_train_prepared = preprocessor.fit_transform(X_train)

X_test_prepared = preprocessor.transform(X_test)

In [13]:
print("Avant préparation :", X_train.shape)
print("Après préparation :", X_train_prepared.shape)

print("Test avant :", X_test.shape)
print("Test après :", X_test_prepared.shape)

Avant préparation : (434, 11)
Après préparation : (434, 191)
Test avant : (109, 11)
Test après : (109, 191)


## Modèle 1 — DummyRegressor

Le `DummyRegressor` sert de modèle de référence.

Il effectue une prédiction très simple sans réellement apprendre les relations entre les variables.

Il permet de vérifier si les modèles plus avancés font réellement mieux qu'une stratégie naïve.

In [14]:
from sklearn.dummy import DummyRegressor

In [15]:
dummy_model = DummyRegressor(strategy="mean")

In [16]:
dummy_model.fit(X_train_prepared, y_train)

,strategy,'mean'
,constant,None
,quantile,None


In [17]:
y_pred_dummy = dummy_model.predict(X_test_prepared)

In [18]:
print("Prix moyen appris :", y_train.mean())

print("Premières prédictions :")
print(y_pred_dummy[:5])

Prix moyen appris : 23760.677419354837
Premières prédictions :
[23760.67741935 23760.67741935 23760.67741935 23760.67741935
 23760.67741935]


## Évaluation du DummyRegressor

Les performances sont évaluées avec trois métriques :

- **MAE** : erreur moyenne entre le prix réel et le prix prédit.
- **RMSE** : similaire à la MAE, mais pénalise davantage les grosses erreurs.
- **R²** : mesure la capacité du modèle à expliquer les variations du prix.

In [19]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

In [20]:
mae_dummy = mean_absolute_error(y_test, y_pred_dummy)
rmse_dummy = root_mean_squared_error(y_test, y_pred_dummy)
r2_dummy = r2_score(y_test, y_pred_dummy)

print("MAE  :", mae_dummy)
print("RMSE :", rmse_dummy)
print("R²   :", r2_dummy)

MAE  : 5631.577389760284
RMSE : 6864.37409126827
R²   : -0.040239363738684863


### Interprétation

Le DummyRegressor obtient une MAE d'environ 5 632 $ et un R² proche de 0.

Ce modèle sert de référence minimale. Les modèles suivants devront obtenir une erreur plus faible et un R² plus élevé pour être considérés comme plus performants.